# Region Sensitivity OpenRouter Evaluation — Prompt-matched Version

This notebook is designed to match the previous Gemini evaluation notebook as closely as possible.

It uses the same official QA JSON file:

```text
region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_1800_eval.json
```

It also constructs the model input using the same prompt structure as the Gemini notebook:

```python
full_prompt = (
    system_prompt
    + "\n\n"
    + question
    + "\n\nImportant instruction: Return only one option letter: A, B, C, or D. "
    + "Do not explain your reasoning."
)
```

Important design choice:

- OpenRouter receives this `full_prompt` as a single user message.
- No extra OpenRouter system prompt is added.
- Gold fields are not sent to the model.
- The answer extraction function is copied from the Gemini notebook.

## 1. Imports and paths

In [22]:
import os
import re
import json
import time
import random
import hashlib
import getpass
from pathlib import Path
from datetime import datetime

import pandas as pd
import requests
from tqdm import tqdm

In [23]:
# -----------------------------
# Paths
# -----------------------------

OUTPUT_DIR = Path("region_sensitivity_outputs")
EVAL_OUTPUT_DIR = Path("region_sensitivity_eval_outputs")
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# This should be the same official QA file used by the Gemini notebook.
OFFICIAL_QA_PATH = OUTPUT_DIR / "region_sensitivity_qa_pairs_context_v2_1800_eval.json"

# Optional debug file if you have it.
DEBUG_QA_PATH = OUTPUT_DIR / "region_sensitivity_qa_pairs_context_v2_debug_15.json"

# Use False for the official 1800-row evaluation.
# The notebook still has a 5-row dry run cell later, so you usually keep this as False.
USE_DEBUG = False

QA_PATH = DEBUG_QA_PATH if USE_DEBUG else OFFICIAL_QA_PATH

print("Official QA path:", OFFICIAL_QA_PATH)
print("Debug QA path:", DEBUG_QA_PATH)
print("Selected QA path:", QA_PATH)
print("Eval output dir:", EVAL_OUTPUT_DIR)

if not QA_PATH.exists():
    raise FileNotFoundError(
        f"Cannot find selected QA file: {QA_PATH.resolve()}\n\n"
        "This prompt-matched OpenRouter notebook expects the official JSON used by the Gemini notebook.\n"
        "Please make sure the file exists, or update OFFICIAL_QA_PATH to the correct absolute path."
    )

Official QA path: region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_1800_eval.json
Debug QA path: region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_debug_15.json
Selected QA path: region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_1800_eval.json
Eval output dir: region_sensitivity_eval_outputs


## 2. Load official QA JSON

This matches the Gemini notebook, which loads the official QA file from JSON.

In [24]:
# -----------------------------
# Load QA data
# -----------------------------

with open(QA_PATH, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

eval_df = pd.DataFrame(qa_data)

print("Loaded QA:", eval_df.shape)

if "task_type" in eval_df.columns:
    print("\nTask distribution:")
    print(eval_df["task_type"].value_counts())

if "answer" in eval_df.columns:
    print("\nAnswer distribution by task:")
    print(pd.crosstab(eval_df["task_type"], eval_df["answer"]))

print("\nColumns:")
print(eval_df.columns.tolist())

display(eval_df.head(2))

Loaded QA: (1800, 9)

Task distribution:
task_type
pairwise_comparison     600
top_sensitive_region    600
management_priority     600
Name: count, dtype: int64

Answer distribution by task:
answer                  A    B    C    D
task_type                               
management_priority   162  157  150  131
pairwise_comparison   293  307    0    0
top_sensitive_region  161  144  142  153

Columns:
['id', 'task_type', 'system_prompt', 'question', 'options', 'answer', 'gold_region', 'weather_condition', 'candidate_lgas']


,id,task_type,system_prompt,question,options,answer,gold_region,weather_condition,candidate_lgas
0,rs_context_v2_pairwise_comparison_0001,pairwise_comparison,You are evaluating weather-sensitive urban tra...,Weather condition:\nno_rain+strong_wind+very_h...,"{'A': 'Sydney', 'B': 'Leichhardt'}",B,Leichhardt,no_rain+strong_wind+very_humid+overcast,"[Sydney, Leichhardt]"
1,rs_context_v2_pairwise_comparison_0002,pairwise_comparison,You are evaluating weather-sensitive urban tra...,Weather condition:\nlight_rain+overcast\n\nCan...,"{'A': 'Auburn', 'B': 'Warringah'}",B,Warringah,light_rain+overcast,"[Auburn, Warringah]"


In [4]:
from pathlib import Path

for f in Path("region_sensitivity_outputs").glob("*.json"):
    print(f.name)

region_sensitivity_qa_pairs_balanced_600_eval.json
region_sensitivity_qa_pairs_expanded_1800.json
region_sensitivity_qa_pairs_context_v2_debug_15.json
region_sensitivity_qa_pairs.json
region_sensitivity_qa_pairs_context_v2_1800_eval.json
region_sensitivity_qa_pairs_context_v2_1800.json
region_sensitivity_qa_pairs_context_v2_balanced_600_eval.json


## 3. Check required fields

These are the same required fields as the Gemini notebook.

In [5]:
# -----------------------------
# Check evaluation fields
# -----------------------------

required_cols = [
    "id",
    "task_type",
    "system_prompt",
    "question",
    "answer",
    "gold_region"
]

missing_cols = [col for col in required_cols if col not in eval_df.columns]

if missing_cols:
    raise ValueError(
        f"Missing columns: {missing_cols}\n\n"
        "This means the selected QA file is not the same official JSON format used by the Gemini evaluation.\n"
        "Do not use the simplified CSV for prompt-matched multi-model evaluation."
    )

print("All required columns found.")
print("Columns in eval_df:")
print(eval_df.columns.tolist())

if not USE_DEBUG:
    assert len(eval_df) == 1800, (
        f"Official evaluation should use 1800 QA rows, but eval_df has {len(eval_df)} rows. "
        "Check that USE_DEBUG=False and the official JSON path is correct."
    )

All required columns found.
Columns in eval_df:
['id', 'task_type', 'system_prompt', 'question', 'options', 'answer', 'gold_region', 'weather_condition', 'candidate_lgas']


## 4. Build prompt exactly like the Gemini notebook

In [25]:
# -----------------------------
# Prompt construction
# -----------------------------

def build_full_prompt(system_prompt, question):
    """
    Build the exact same prompt string used in the Gemini notebook.

    Gemini notebook logic:
        full_prompt = (
            system_prompt
            + "\n\n"
            + question
            + "\n\nImportant instruction: Return only one option letter: A, B, C, or D. "
            + "Do not explain your reasoning."
        )
    """
    return (
        str(system_prompt)
        + "\n\n"
        + str(question)
        + "\n\nImportant instruction: Return only one option letter: A, B, C, or D. "
        + "Do not explain your reasoning."
    )


# Check the first prompt.
test_full_prompt = build_full_prompt(
    eval_df.iloc[0]["system_prompt"],
    eval_df.iloc[0]["question"]
)

print("First full prompt SHA256:")
print(hashlib.sha256(test_full_prompt.encode("utf-8")).hexdigest())

print("\nFirst full prompt preview:")
print(test_full_prompt[:3000])

print("\nGold fields are NOT included in the prompt.")
print("Gold answer for row 0, shown only for checking:", eval_df.iloc[0]["answer"])
print("Gold region for row 0, shown only for checking:", eval_df.iloc[0]["gold_region"])

First full prompt SHA256:
2a1d036b4a0d3aafbb726c5dac5fa74cfff7d0abc9781dc0a931799bd8679a81

First full prompt preview:
You are evaluating weather-sensitive urban traffic patterns using the provided regional context and traffic evidence. Use only the information provided in the question. Do not rely on external assumptions about the LGA names. A weather-sensitive traffic region is one whose observed traffic appears to deviate more strongly or more frequently from its expected traffic level under the given weather condition. Return only one option letter from the given options.

Weather condition:
no_rain+strong_wind+very_humid+overcast

Candidate LGAs:

A. Sydney
- Number of traffic stations: 12
- Average expected traffic volume: 1088.1
- Average observed traffic volume under this weather condition: 1052.1
- Observed traffic compared with typical level: lower than typical
- Dominant land-use type: residential
- POI density: high
- Significant traffic change frequency: high
- Typical tra

## 5. Answer extraction

This function is copied from the Gemini notebook to keep evaluation consistent.

In [26]:
# -----------------------------
# Extract option letter from model response
# -----------------------------

def extract_option_letter(response_text):
    """
    Extract A/B/C/D from model output.

    This function avoids taking the first option letter mentioned in the reasoning.
    It prioritises final-answer patterns such as:
    - "The final answer is C"
    - "Answer: C"
    - "\\boxed{C}"
    """

    if response_text is None:
        return None

    text = str(response_text).strip()
    upper_text = text.upper().strip()

    # Direct one-letter answer, allowing punctuation
    direct_match = re.match(r"^\s*([ABCD])\s*[\.\)]?\s*$", upper_text)
    if direct_match:
        return direct_match.group(1)

    # Normalise whitespace
    upper_text = re.sub(r"\s+", " ", upper_text)

    # LaTeX boxed answer, e.g. \boxed{C}
    boxed_match = re.search(r"\\?BOXED\{([ABCD])\}", upper_text)
    if boxed_match:
        return boxed_match.group(1)

    # Common final-answer patterns
    final_patterns = [
        r"THE FINAL ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"FINAL ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"FINAL ANSWER\s*[:\-]?\s*([ABCD])",
        r"THE ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"ANSWER\s*[:\-]?\s*([ABCD])",
        r"OPTION\s*([ABCD])",
        r"CHOOSE\s*([ABCD])",
    ]

    for pattern in final_patterns:
        matches = re.findall(pattern, upper_text)
        if matches:
            return matches[-1]

    # Fallback: use the last standalone A/B/C/D, not the first.
    # This handles outputs that discuss all options before giving a final answer.
    all_matches = re.findall(r"\b([ABCD])\b", upper_text)

    if all_matches:
        return all_matches[-1]

    return None


# Quick test
test_outputs = [
    "A",
    "A.",
    "Option B",
    "The answer is C.",
    "D because ...",
    "The final answer is \\boxed{C}",
    "After comparing A, B, C, and D, the final answer is C.",
    "unknown"
]

for x in test_outputs:
    print(x, "->", extract_option_letter(x))

A -> A
A. -> A
Option B -> B
The answer is C. -> C
D because ... -> D
The final answer is \boxed{C} -> C
After comparing A, B, C, and D, the final answer is C. -> C
unknown -> None


## 6. OpenRouter configuration

Default model:

```text
qwen/qwen3-32b
```

This is the paid / normal endpoint. The free endpoint would have `:free` at the end, but this notebook defaults to the paid model for official evaluation.

In [45]:
# -----------------------------
# OpenRouter configuration
# -----------------------------

OPENROUTER_API_URL = "https://openrouter.ai/api/v1/chat/completions"

MODEL_PROVIDER = "openrouter"

# Paid / normal endpoint:
MODEL_NAME = "qwen/qwen3-32b"

# Free endpoint, not recommended for official 1800-row evaluation:
# MODEL_NAME = "meta-llama/llama-3.3-70b-instruct:free"

MODEL_SLUG = MODEL_NAME.replace("/", "_").replace(":", "_")

# Use a new filename to avoid mixing with the previous non-prompt-matched dry run.
SAVE_PATH = EVAL_OUTPUT_DIR / f"results_{MODEL_SLUG}_context_v2_1800_promptmatched.csv"

# For deterministic classification-style evaluation across OpenRouter models.
# Note: Gemini notebook did not explicitly set decoding parameters.
# For OpenRouter models, keep these constant across all OpenRouter evaluations.
TEMPERATURE = 0
MAX_TOKENS = 1000

# Delay to reduce rate-limit risk.
SLEEP_SECONDS = 3.0

print("Model provider:", MODEL_PROVIDER)
print("Model name:", MODEL_NAME)
print("Save path:", SAVE_PATH)

Model provider: openrouter
Model name: qwen/qwen3-32b
Save path: region_sensitivity_eval_outputs/results_qwen_qwen3-32b_context_v2_1800_promptmatched.csv


## 7. Set OpenRouter API key

Do not hard-code your key into the notebook.

In [28]:
# -----------------------------
# API key
# -----------------------------

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass.getpass("Enter your OpenRouter API key: ").strip()

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is empty.")

print("OpenRouter API key loaded.")

Enter your OpenRouter API key:  ········


OpenRouter API key loaded.


## 8. OpenRouter call function

Important: no extra system message is added.

The model receives only the Gemini-style `full_prompt` as one user message.

In [46]:
# -----------------------------
# OpenRouter API call
# -----------------------------

def should_stop_run(status_code, response_text):
    """
    Detect errors where continuing would waste time or create many failed rows.
    """
    text = (response_text or "").lower()

    stop_phrases = [
        "invalid api key",
        "no auth credentials",
        "unauthorized",
        "forbidden",
        "insufficient credits",
        "credits",
        "quota",
        "rate limit",
        "too many requests",
        "payment required",
    ]

    if status_code in [401, 402, 403]:
        return True

    if any(phrase in text for phrase in stop_phrases):
        return True

    return False

# ============================================================
# Improved OpenRouter call with empty-content retry
# ============================================================

import time
import random
import requests


def call_openrouter_model(
    full_prompt,
    model_name=MODEL_NAME,
    max_retries=6
):
    """
    Call OpenRouter and return a non-empty text response.

    HTTP 200 with empty message content is treated as a transient
    failure and retried instead of being recorded as a successful row.
    """

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://chat.openai.com/",
        "X-Title": "Region Sensitivity Prompt-Matched Evaluation",
    }

    payload = {
        "model": model_name,
        "messages": [
            {
                "role": "user",
                "content": full_prompt
            }
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
    }

    last_error = None

    for attempt in range(1, max_retries + 1):

        try:
            response = requests.post(
                OPENROUTER_API_URL,
                headers=headers,
                json=payload,
                timeout=180,
            )

            # --------------------------------------------
            # Successful HTTP response
            # --------------------------------------------
            if response.status_code == 200:
                data = response.json()

                choices = data.get("choices", [])

                if not choices:
                    last_error = (
                        "HTTP 200 but response contained no choices."
                    )

                else:
                    choice = choices[0]
                    message = choice.get("message", {})
                    content = message.get("content")
                    finish_reason = choice.get("finish_reason")

                    # Some APIs may return content as a list
                    if isinstance(content, list):
                        text_parts = []

                        for item in content:
                            if isinstance(item, dict):
                                item_text = item.get("text")
                                if item_text:
                                    text_parts.append(str(item_text))
                            elif item is not None:
                                text_parts.append(str(item))

                        content = "".join(text_parts)

                    # Valid non-empty response
                    if content is not None and str(content).strip():
                        return str(content).strip()

                    # Diagnostic information for empty content
                    reasoning = (
                        message.get("reasoning")
                        or message.get("reasoning_content")
                        or ""
                    )

                    last_error = (
                        "HTTP 200 but model returned empty content. "
                        f"finish_reason={finish_reason}; "
                        f"reasoning_length={len(str(reasoning))}"
                    )

                if attempt < max_retries:
                    wait_time = min(
                        60,
                        (2 ** attempt) + random.random()
                    )

                    print(
                        f"Empty response. Retry "
                        f"{attempt}/{max_retries} "
                        f"after {wait_time:.1f}s."
                    )
                    print("Details:", last_error)

                    time.sleep(wait_time)
                    continue

                raise RuntimeError(last_error)

            # --------------------------------------------
            # Non-200 response
            # --------------------------------------------
            response_text = response.text

            last_error = (
                f"HTTP {response.status_code}: "
                f"{response_text[:1000]}"
            )

            if should_stop_run(
                response.status_code,
                response_text
            ):
                raise RuntimeError(
                    "STOP_RUN: OpenRouter quota/auth/credit/"
                    "rate-limit problem. "
                    f"Details: {last_error}"
                )

            if response.status_code in [
                429,
                500,
                502,
                503,
                504,
            ]:
                wait_time = min(
                    60,
                    (2 ** attempt) + random.random()
                )

                print(
                    f"Transient HTTP error. Retry "
                    f"{attempt}/{max_retries} "
                    f"after {wait_time:.1f}s."
                )

                time.sleep(wait_time)
                continue

            raise RuntimeError(last_error)

        except requests.exceptions.RequestException as e:
            last_error = repr(e)

            if attempt < max_retries:
                wait_time = min(
                    60,
                    (2 ** attempt) + random.random()
                )

                print(
                    f"Request exception. Retry "
                    f"{attempt}/{max_retries} "
                    f"after {wait_time:.1f}s."
                )

                time.sleep(wait_time)
                continue

            raise RuntimeError(
                "OpenRouter request failed after retries: "
                f"{last_error}"
            )

    raise RuntimeError(
        f"OpenRouter request failed after retries: {last_error}"
    )

## 9. Resume helpers

Successful rows are skipped.

Previous error rows are intentionally rerun.

In [33]:
# -----------------------------
# Resume helpers
# -----------------------------

def is_successful_result(row):
    error_value = row.get("error", None)
    pred_value = row.get("predicted_answer", None)

    error_empty = pd.isna(error_value) or str(error_value).strip() == ""
    pred_valid = str(pred_value).strip().upper() in ["A", "B", "C", "D"]

    return error_empty and pred_valid


def load_existing_successes(save_path):
    """
    Load existing output and keep only successful rows.
    Error rows are dropped so they can be rerun.
    """
    if not Path(save_path).exists():
        print("No existing result file found. Starting fresh.")
        return pd.DataFrame(), set()

    existing_df = pd.read_csv(save_path)

    if len(existing_df) == 0:
        return existing_df, set()

    if "id" not in existing_df.columns:
        raise ValueError("Existing output file does not contain 'id'. Check file compatibility.")

    success_mask = existing_df.apply(is_successful_result, axis=1)
    success_df = existing_df[success_mask].copy()
    success_ids = set(success_df["id"].astype(str).tolist())

    print(f"Existing rows: {len(existing_df)}")
    print(f"Existing successful rows kept: {len(success_df)}")
    print(f"Previous error rows to rerun: {len(existing_df) - len(success_df)}")

    return success_df, success_ids


def save_results(results, save_path=SAVE_PATH):
    result_df = pd.DataFrame(results)

    if "id" in result_df.columns:
        result_df = result_df.sort_values(by="id").reset_index(drop=True)

    result_df.to_csv(save_path, index=False)
    return result_df

## 10. Main evaluation function

Use `run_limit=5` first.

Then use `run_limit=None` for the full remaining run.

In [34]:
# -----------------------------
# Main evaluation
# -----------------------------

def run_evaluation(run_limit=5, sleep_seconds=SLEEP_SECONDS):
    """
    Run OpenRouter evaluation with prompt-matched input and safe resume.

    Args:
        run_limit:
            5 for dry run.
            None for all remaining rows.
        sleep_seconds:
            Delay between successful requests.
    """

    success_df, success_ids = load_existing_successes(SAVE_PATH)

    work_df = eval_df.copy()
    work_df["id"] = work_df["id"].astype(str)

    rows_to_run = work_df[~work_df["id"].isin(success_ids)].copy()

    # Keep the original official QA order to match the Gemini evaluation order as closely as possible.
    if run_limit is not None:
        rows_to_run = rows_to_run.head(run_limit).copy()

    print("\nEvaluation setup")
    print("----------------")
    print("Model provider:", MODEL_PROVIDER)
    print("Model name:", MODEL_NAME)
    print("Total QA rows:", len(eval_df))
    print("Already successful:", len(success_ids))
    print("Rows to run now:", len(rows_to_run))
    print("Saving to:", SAVE_PATH)

    results = success_df.to_dict("records")

    if len(rows_to_run) == 0:
        print("No remaining rows to run.")
        return save_results(results, SAVE_PATH)

    for local_i, (_, row) in enumerate(rows_to_run.iterrows(), start=1):
        print(
            f"\n[{local_i}/{len(rows_to_run)}] "
            f"id={row['id']} task={row['task_type']} gold={row['answer']}"
        )

        raw_response = None
        predicted_answer = None
        error = None

        try:
            full_prompt = build_full_prompt(
                row["system_prompt"],
                row["question"]
            )

            raw_response = call_openrouter_model(
                full_prompt=full_prompt,
                model_name=MODEL_NAME,
                max_retries=3
            )

            predicted_answer = extract_option_letter(raw_response)

            result_row = {
                "id": row["id"],
                "task_type": row["task_type"],
                "weather_condition": row.get("weather_condition", None),
                "model_provider": MODEL_PROVIDER,
                "model_name": MODEL_NAME,
                "gold_answer": row["answer"],
                "gold_region": row["gold_region"],
                "raw_response": raw_response,
                "predicted_answer": predicted_answer,
                "is_correct": predicted_answer == row["answer"],
                "error": None,
                "started_at": datetime.now().isoformat(timespec="seconds"),
                "finished_at": datetime.now().isoformat(timespec="seconds"),
            }

            print("Raw response:", repr(raw_response))
            print("Parsed:", predicted_answer, "| Correct:", result_row["is_correct"])

        except Exception as e:
            error = str(e)

            result_row = {
                "id": row["id"],
                "task_type": row["task_type"],
                "weather_condition": row.get("weather_condition", None),
                "model_provider": MODEL_PROVIDER,
                "model_name": MODEL_NAME,
                "gold_answer": row["answer"],
                "gold_region": row["gold_region"],
                "raw_response": None,
                "predicted_answer": None,
                "is_correct": False,
                "error": error,
                "started_at": datetime.now().isoformat(timespec="seconds"),
                "finished_at": datetime.now().isoformat(timespec="seconds"),
            }

            print("Error:", error)

            if "STOP_RUN" in error:
                print("\nStopping evaluation to avoid writing many failed rows.")
                break

        results.append(result_row)

        # Save after every row to avoid losing progress.
        save_results(results, SAVE_PATH)

        print(f"Saved rows: {len(results)}")

        if error is None:
            time.sleep(sleep_seconds)

    results_df = save_results(results, SAVE_PATH)

    print("\nRun finished or stopped.")
    print("Saved results to:", SAVE_PATH)
    print("Total saved rows:", len(results_df))
    print("Number of errors:", results_df["error"].notna().sum())

    if len(results_df) > 0:
        print("\nOverall accuracy including errors as wrong:", results_df["is_correct"].mean())
        print("\nAccuracy by task:")
        print(results_df.groupby("task_type")["is_correct"].mean())

    return results_df

## 11. Dry run: 5 rows

Run this first.

If `predicted_answer` is parsed as A/B/C/D and there are no errors, continue to the full run.

In [17]:
dry_df = run_evaluation(run_limit=5, sleep_seconds=SLEEP_SECONDS)
display(dry_df.tail())

No existing result file found. Starting fresh.

Evaluation setup
----------------
Model provider: openrouter
Model name: qwen/qwen3-32b
Total QA rows: 1800
Already successful: 0
Rows to run now: 5
Saving to: region_sensitivity_eval_outputs/results_qwen_qwen3-32b_context_v2_1800_promptmatched.csv

[1/5] id=rs_context_v2_pairwise_comparison_0001 task=pairwise_comparison gold=B
Raw response: 'B'
Parsed: B | Correct: True
Saved rows: 1

[2/5] id=rs_context_v2_pairwise_comparison_0002 task=pairwise_comparison gold=B
Raw response: 'B'
Parsed: B | Correct: True
Saved rows: 2

[3/5] id=rs_context_v2_pairwise_comparison_0003 task=pairwise_comparison gold=B
Raw response: 'B'
Parsed: B | Correct: True
Saved rows: 3

[4/5] id=rs_context_v2_pairwise_comparison_0004 task=pairwise_comparison gold=B
Raw response: 'B'
Parsed: B | Correct: True
Saved rows: 4

[5/5] id=rs_context_v2_pairwise_comparison_0005 task=pairwise_comparison gold=B
Raw response: 'B'
Parsed: B | Correct: True
Saved rows: 5

Run fin

,id,task_type,weather_condition,model_provider,model_name,gold_answer,gold_region,raw_response,predicted_answer,is_correct,error,started_at,finished_at
0,rs_context_v2_pairwise_comparison_0001,pairwise_comparison,no_rain+strong_wind+very_humid+overcast,openrouter,qwen/qwen3-32b,B,Leichhardt,B,B,True,None,2026-07-10T12:49:00,2026-07-10T12:49:00
1,rs_context_v2_pairwise_comparison_0002,pairwise_comparison,light_rain+overcast,openrouter,qwen/qwen3-32b,B,Warringah,B,B,True,None,2026-07-10T12:49:10,2026-07-10T12:49:10
2,rs_context_v2_pairwise_comparison_0003,pairwise_comparison,heavy_rain+very_humid+overcast,openrouter,qwen/qwen3-32b,B,Parramatta,B,B,True,None,2026-07-10T12:49:27,2026-07-10T12:49:27
3,rs_context_v2_pairwise_comparison_0004,pairwise_comparison,no_rain+extreme_heat,openrouter,qwen/qwen3-32b,B,Parramatta,B,B,True,None,2026-07-10T12:49:36,2026-07-10T12:49:36
4,rs_context_v2_pairwise_comparison_0005,pairwise_comparison,no_rain+very_humid,openrouter,qwen/qwen3-32b,B,Warringah,B,B,True,None,2026-07-10T12:49:41,2026-07-10T12:49:41


## 13. Check dry-run summary

In [18]:
if SAVE_PATH.exists():
    check_df = pd.read_csv(SAVE_PATH)

    print("Rows saved:", len(check_df))
    print("Errors:", check_df["error"].notna().sum())

    print("\nPrediction distribution:")
    print(check_df["predicted_answer"].value_counts(dropna=False).sort_index())

    print("\nAccuracy so far:")
    print(check_df["is_correct"].mean())

    print("\nAccuracy by task:")
    print(check_df.groupby("task_type")["is_correct"].mean())

Rows saved: 5
Errors: 0

Prediction distribution:
predicted_answer
B    5
Name: count, dtype: int64

Accuracy so far:
1.0

Accuracy by task:
task_type
pairwise_comparison    1.0
Name: is_correct, dtype: float64


## 14. Full run: all remaining rows

Only run this after the 5-row dry run looks correct.

This will resume from the existing output file and skip successful rows.

In [57]:
final_qwen_df = run_evaluation(
    run_limit=None,
    sleep_seconds=SLEEP_SECONDS
)

Existing rows: 1800
Existing successful rows kept: 1799
Previous error rows to rerun: 1

Evaluation setup
----------------
Model provider: openrouter
Model name: qwen/qwen3-32b
Total QA rows: 1800
Already successful: 1799
Rows to run now: 1
Saving to: region_sensitivity_eval_outputs/results_qwen_qwen3-32b_context_v2_1800_promptmatched.csv

[1/1] id=rs_context_v2_management_priority_0242 task=management_priority gold=D
Empty response. Retry 1/3 after 2.9s.
Details: HTTP 200 but model returned empty content. finish_reason=length; reasoning_length=3937
Raw response: 'D'
Parsed: D | Correct: True
Saved rows: 1800

Run finished or stopped.
Saved results to: region_sensitivity_eval_outputs/results_qwen_qwen3-32b_context_v2_1800_promptmatched.csv
Total saved rows: 1800
Number of errors: 0

Overall accuracy including errors as wrong: 0.6205555555555555

Accuracy by task:
task_type
management_priority     0.563333
pairwise_comparison     0.753333
top_sensitive_region    0.545000
Name: is_correc

## 15. Final summary

## 16. Save compact summary CSV

In [21]:
# -----------------------------
# Save compact model summary
# -----------------------------

summary_rows = []

results_df = pd.read_csv(SAVE_PATH)
results_df["error_clean"] = results_df["error"].fillna("").astype(str).str.strip()
results_df["success"] = results_df["error_clean"] == ""
results_df["is_correct_bool"] = results_df["is_correct"].astype(str).str.lower().isin(["true", "1", "yes"])

valid_df = results_df[results_df["success"]].copy()

summary_rows.append({
    "model_provider": MODEL_PROVIDER,
    "model_name": MODEL_NAME,
    "task_type": "overall",
    "total_rows": len(results_df),
    "successful_rows": int(results_df["success"].sum()),
    "error_rows": int((~results_df["success"]).sum()),
    "accuracy_valid_only": valid_df["is_correct_bool"].mean() if len(valid_df) > 0 else None,
    "accuracy_errors_as_wrong": results_df["is_correct_bool"].mean() if len(results_df) > 0 else None,
    "output_file": str(SAVE_PATH),
})

for task_value, sub in results_df.groupby("task_type"):
    sub_valid = sub[sub["success"]].copy()

    summary_rows.append({
        "model_provider": MODEL_PROVIDER,
        "model_name": MODEL_NAME,
        "task_type": task_value,
        "total_rows": len(sub),
        "successful_rows": int(sub["success"].sum()),
        "error_rows": int((~sub["success"]).sum()),
        "accuracy_valid_only": sub_valid["is_correct_bool"].mean() if len(sub_valid) > 0 else None,
        "accuracy_errors_as_wrong": sub["is_correct_bool"].mean() if len(sub) > 0 else None,
        "output_file": str(SAVE_PATH),
    })

model_summary_df = pd.DataFrame(summary_rows)

SUMMARY_PATH = (
    EVAL_OUTPUT_DIR
    / "model_accuracy_summary_openrouter_qwen3_32b_promptmatched.csv"
)

model_summary_df.to_csv(SUMMARY_PATH, index=False)

print("Saved summary:", SUMMARY_PATH)
print("Absolute path:", SUMMARY_PATH.resolve())
print("Exists:", SUMMARY_PATH.exists())

display(model_summary_df)

Saved summary: region_sensitivity_eval_outputs/model_accuracy_summary_openrouter_qwen3_32b_promptmatched.csv
Absolute path: /Users/tanghuiru/Desktop/labeled_data_v1/region_sensitivity_eval_outputs/model_accuracy_summary_openrouter_qwen3_32b_promptmatched.csv
Exists: True


,model_provider,model_name,task_type,total_rows,successful_rows,error_rows,accuracy_valid_only,accuracy_errors_as_wrong,output_file
0,openrouter,qwen/qwen3-32b,overall,1800,1800,0,0.563333,0.563333,region_sensitivity_eval_outputs/results_qwen_q...
1,openrouter,qwen/qwen3-32b,management_priority,600,600,0,0.520000,0.520000,region_sensitivity_eval_outputs/results_qwen_q...
2,openrouter,qwen/qwen3-32b,pairwise_comparison,600,600,0,0.703333,0.703333,region_sensitivity_eval_outputs/results_qwen_q...
3,openrouter,qwen/qwen3-32b,top_sensitive_region,600,600,0,0.466667,0.466667,region_sensitivity_eval_outputs/results_qwen_q...


In [1]:
# ============================================================
# Task 5 metric verification: Qwen3 32B
# ============================================================

import os
import glob
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score
)

# ------------------------------------------------------------
# 1. Automatically find Qwen3 result CSV files
# ------------------------------------------------------------

output_folder = "region_sensitivity_eval_outputs"

qwen_files = sorted(
    glob.glob(
        os.path.join(
            output_folder,
            "*qwen3-32b*promptmatched*.csv"
        )
    ),
    key=os.path.getmtime,
    reverse=True
)

print("Qwen3 files found:")

for i, file_path in enumerate(qwen_files):
    print(
        i,
        os.path.basename(file_path),
        "| modified:",
        pd.Timestamp(os.path.getmtime(file_path), unit="s")
    )

if len(qwen_files) == 0:
    raise FileNotFoundError(
        "No Qwen3 32B promptmatched CSV was found."
    )

# Use the most recently modified matching CSV
file_path = qwen_files[0]

print("\nSelected file:")
print(file_path)


# ------------------------------------------------------------
# 2. Load results
# ------------------------------------------------------------

df = pd.read_csv(file_path)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 3. Standardise answers and task types
# ------------------------------------------------------------

def clean_answer(value):
    if pd.isna(value):
        return None

    answer = str(value).strip().upper()

    if answer in ["A", "B", "C", "D"]:
        return answer

    return None


df["gold_clean"] = df["gold_answer"].apply(clean_answer)
df["pred_clean"] = df["predicted_answer"].apply(clean_answer)

df["task_clean"] = (
    df["task_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

valid_df = df[
    df["gold_clean"].isin(["A", "B", "C", "D"])
    & df["pred_clean"].isin(["A", "B", "C", "D"])
].copy()


# ------------------------------------------------------------
# 4. Basic checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BASIC CHECK")
print("=" * 70)

print("Total rows:", len(df))
print("Valid rows:", len(valid_df))
print("Invalid or empty predictions:", len(df) - len(valid_df))

if "error" in df.columns:
    error_count = (
        df["error"].notna()
        & df["error"].astype(str).str.strip().ne("")
    ).sum()

    print("Rows with error:", error_count)


# ------------------------------------------------------------
# 5. Overall Accuracy and Macro-F1
# ------------------------------------------------------------

y_true = valid_df["gold_clean"]
y_pred = valid_df["pred_clean"]

overall_accuracy = accuracy_score(
    y_true,
    y_pred
)

overall_macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=["A", "B", "C", "D"],
    average="macro",
    zero_division=0
)

print("\n" + "=" * 70)
print("OVERALL RESULTS")
print("=" * 70)

print(f"Accuracy check: {overall_accuracy * 100:.2f}%")
print(f"Overall Macro-F1: {overall_macro_f1 * 100:.2f}%")


# ------------------------------------------------------------
# 6. Accuracy and Macro-F1 by question type
# ------------------------------------------------------------

task_settings = {
    "pairwise_comparison": ["A", "B"],
    "top_sensitive_region": ["A", "B", "C", "D"],
    "management_priority": ["A", "B", "C", "D"]
}

task_rows = []

for task_name, labels in task_settings.items():

    task_df = valid_df[
        valid_df["task_clean"] == task_name
    ]

    task_accuracy = accuracy_score(
        task_df["gold_clean"],
        task_df["pred_clean"]
    )

    task_macro_f1 = f1_score(
        task_df["gold_clean"],
        task_df["pred_clean"],
        labels=labels,
        average="macro",
        zero_division=0
    )

    task_rows.append({
        "Task Type": task_name,
        "Valid Rows": len(task_df),
        "Accuracy (%)": round(task_accuracy * 100, 2),
        "Macro-F1 (%)": round(task_macro_f1 * 100, 2)
    })

task_table = pd.DataFrame(task_rows)

print("\n" + "=" * 70)
print("RESULTS BY QUESTION TYPE")
print("=" * 70)

display(task_table)


# ------------------------------------------------------------
# 7. Option-level Recall and F1
# ------------------------------------------------------------

options = ["A", "B", "C", "D"]

recall_values = recall_score(
    y_true,
    y_pred,
    labels=options,
    average=None,
    zero_division=0
)

f1_values = f1_score(
    y_true,
    y_pred,
    labels=options,
    average=None,
    zero_division=0
)

option_table = pd.DataFrame({
    "Option": options,
    "Recall (%)": [
        round(value * 100, 2)
        for value in recall_values
    ],
    "F1 (%)": [
        round(value * 100, 2)
        for value in f1_values
    ]
})

print("\n" + "=" * 70)
print("OPTION-LEVEL RECALL AND F1")
print("=" * 70)

display(option_table)


# ------------------------------------------------------------
# 8. Prediction distribution
# ------------------------------------------------------------

prediction_counts = (
    y_pred.value_counts()
    .reindex(options, fill_value=0)
)

distribution_table = pd.DataFrame({
    "Option": options,
    "Prediction Count": [
        int(prediction_counts[option])
        for option in options
    ],
    "Prediction Percentage (%)": [
        round(
            prediction_counts[option] / len(valid_df) * 100,
            2
        )
        for option in options
    ]
})

print("\n" + "=" * 70)
print("PREDICTION DISTRIBUTION")
print("=" * 70)

display(distribution_table)

Qwen3 files found:
0 results_qwen_qwen3-32b_context_v2_1800_promptmatched.csv | modified: 2026-07-22 11:50:45.848208904
1 results_qwen_qwen3-32b_context_v2_1800_promptmatched_before_retry_90.csv | modified: 2026-07-15 03:13:34.201175928
2 results_qwen_qwen3-32b_context_v2_1800_promptmatched_before_rerun_backup.csv | modified: 2026-07-10 08:30:07.353705645

Selected file:
region_sensitivity_eval_outputs/results_qwen_qwen3-32b_context_v2_1800_promptmatched.csv

Shape:
(1800, 13)

Columns:
['id', 'task_type', 'weather_condition', 'model_provider', 'model_name', 'gold_answer', 'gold_region', 'raw_response', 'predicted_answer', 'is_correct', 'error', 'started_at', 'finished_at']

BASIC CHECK
Total rows: 1800
Valid rows: 1800
Invalid or empty predictions: 0
Rows with error: 0

OVERALL RESULTS
Accuracy check: 62.06%
Overall Macro-F1: 60.28%

RESULTS BY QUESTION TYPE


,Task Type,Valid Rows,Accuracy (%),Macro-F1 (%)
0,pairwise_comparison,600,75.33,75.31
1,top_sensitive_region,600,54.50,54.31
2,management_priority,600,56.33,56.23



OPTION-LEVEL RECALL AND F1


,Option,Recall (%),F1 (%)
0,A,66.88,65.19
1,B,63.82,64.94
2,C,58.22,56.76
3,D,51.76,54.24



PREDICTION DISTRIBUTION


,Option,Prediction Count,Prediction Percentage (%)
0,A,648,36.00
1,B,587,32.61
2,C,307,17.06
3,D,258,14.33
